In [31]:

import pandas as pd
import os
from google.colab import files

# ==========================================
# 1. LOAD RAW DATA
# ==========================================
input_file = "/content/retail-orders-raw.csv"

df = pd.read_csv(input_file)

print("=" * 45)
print("RAW DATA")
print("=" * 45)
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

# ==========================================
# 2. BEFORE CLEANING REPORT
# ==========================================
print("\nBEFORE CLEANING")
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

# ==========================================
# 3. STANDARDIZE COLUMN NAMES AND TEXT
# ==========================================
clean_df = df.copy()

clean_df.columns = clean_df.columns.str.strip().str.lower()

text_columns = [
    "order_id", "customer_segment", "city",
    "category", "payment_status"
]

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Standardize text capitalization
for col in ["customer_segment", "payment_status", "city", "category"]:
    clean_df[col] = clean_df[col].str.title()

# ==========================================
# 4. REMOVE DUPLICATES
# ==========================================
before_duplicates = len(clean_df)

clean_df = clean_df.drop_duplicates()

print("\nExact duplicate rows removed:",
      before_duplicates - len(clean_df))

# Remove duplicate order IDs, keeping the first record
before_ids = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["order_id"], keep="first"
)

print("Duplicate order-ID records removed:",
      before_ids - len(clean_df))

# ==========================================
# 5. CORRECT DATA TYPES
# ==========================================
clean_df["order_date"] = pd.to_datetime(
    clean_df["order_date"],
    errors="coerce",
    dayfirst=True
)

for col in ["quantity", "unit_price", "discount_pct"]:
    clean_df[col] = pd.to_numeric(
        clean_df[col], errors="coerce"
    )

# ==========================================
# 6. HANDLE INVALID DATES
# ==========================================
print("\nInvalid or missing dates:",
      clean_df["order_date"].isna().sum())

# Remove rows whose dates cannot be verified
clean_df = clean_df.dropna(subset=["order_date"])

# ==========================================
# 7. HANDLE MISSING AND INVALID VALUES
# ==========================================

# Fill missing city
clean_df["city"] = clean_df["city"].fillna("Unknown")

# Replace invalid discounts with missing values
invalid_discount = (
    (clean_df["discount_pct"] < 0) |
    (clean_df["discount_pct"] > 100)
)
clean_df.loc[invalid_discount, "discount_pct"] = pd.NA

# Fill missing discounts with zero
clean_df["discount_pct"] = clean_df["discount_pct"].fillna(0)

# Mark invalid quantities as missing
clean_df.loc[clean_df["quantity"] <= 0, "quantity"] = pd.NA

# Fill missing/invalid quantities using the median of valid quantities
valid_quantity = clean_df.loc[
    clean_df["quantity"] > 0, "quantity"
]

if not valid_quantity.empty:
    median_quantity = valid_quantity.median()
    clean_df["quantity"] = clean_df["quantity"].fillna(
        median_quantity
    )
else:
    raise ValueError("No valid positive quantities found.")

# Remove records with missing essential values
clean_df = clean_df.dropna(
    subset=["order_id", "unit_price"]
)

# Remove negative unit prices
clean_df = clean_df[clean_df["unit_price"] >= 0]

# Use whole-number quantities for this retail dataset
clean_df["quantity"] = clean_df["quantity"].round().astype("int64")

# ==========================================
# 8. FEATURE ENGINEERING
# ==========================================

clean_df["year"] = clean_df["order_date"].dt.year
clean_df["month"] = clean_df["order_date"].dt.month
clean_df["month_name"] = clean_df["order_date"].dt.month_name()
clean_df["quarter"] = "Q" + clean_df["order_date"].dt.quarter.astype(str)

# Calculate sales metrics
clean_df["gross_sales"] = (
    clean_df["quantity"] * clean_df["unit_price"]
)

clean_df["discount_amount"] = (
    clean_df["gross_sales"] *
    clean_df["discount_pct"] / 100
)

clean_df["net_sales"] = (
    clean_df["gross_sales"] - clean_df["discount_amount"]
)

# Profit and profit margin are not calculated because
# product cost data is not available.

# ==========================================
# 9. FINAL VALIDATION
# ==========================================

print("\n" + "=" * 45)
print("AFTER CLEANING")
print("=" * 45)

print("Final shape:", clean_df.shape)
print("\nMissing values:")
print(clean_df.isnull().sum())
print("\nTotal missing values:",
      clean_df.isnull().sum().sum())
print("\nDuplicate rows:", clean_df.duplicated().sum())
print("Duplicate order IDs:",
      clean_df["order_id"].duplicated().sum())
print("Invalid dates:", clean_df["order_date"].isna().sum())
print("Invalid quantities:", (clean_df["quantity"] <= 0).sum())
print(
    "Invalid discounts:",
    (
        (clean_df["discount_pct"] < 0) |
        (clean_df["discount_pct"] > 100)
    ).sum()
)

print("\nData types:")
print(clean_df.dtypes)

print("\nClean dataset preview:")
display(clean_df)

# ==========================================
# 10. BEFORE VS AFTER SUMMARY
# ==========================================

print("\n" + "=" * 45)
print("BEFORE VS AFTER")
print("=" * 45)
print("Original rows:", len(df))
print("Final rows:", len(clean_df))
print("Original columns:", len(df.columns))
print("Final columns:", len(clean_df.columns))
print("Original missing values:", df.isnull().sum().sum())
print("Final missing values:", clean_df.isnull().sum().sum())

# ==========================================
# 11. EXPORT CLEAN DATASET
# ==========================================

output_file = "/content/clean_dataset.csv"

clean_df.to_csv(output_file, index=False)

print("\n" + "=" * 45)
print("CLEANING PROCESS FINISHED")
print("=" * 45)
print("CSV file created:", os.path.exists(output_file))
print("Output file:", output_file)

# Download CSV
files.download(output_file)


RAW DATA
Rows and columns: (12, 9)

Columns:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

First 5 rows:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid



BEFORE CLEANING

Missing values:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64

Duplicate rows: 1
Duplicate order IDs: 1

Exact duplicate rows removed: 1
Duplicate order-ID records removed: 0

Invalid or missing dates: 5

AFTER CLEANING
Final shape: (6, 16)

Missing values:
order_id            0
order_date          0
customer_segment    0
city                0
category            0
quantity            0
unit_price          0
discount_pct        0
payment_status      0
year                0
month               0
month_name          0
quarter             0
gross_sales         0
discount_amount     0
net_sales           0
dtype: int64

Total missing values: 0

Duplicate rows: 0
Duplicate order IDs: 0
Invalid dates: 0
Invalid quantities: 0
Invalid discounts: 0

Data types:
order_id            string[python]
order_date         

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,year,month,month_name,quarter,gross_sales,discount_amount,net_sales
0,RT-1001,2026-03-01,Student,Chennai,Learning Kit,2,799,10.0,Paid,2026,3,March,Q1,1598,159.80,1438.20
2,RT-1003,2026-05-01,Student,Chennai,Course Access,1,1499,0.0,Pending,2026,5,May,Q2,1499,0.00,1499.00
3,RT-1004,2026-07-01,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2026,7,July,Q3,2397,119.85,2277.15
5,RT-1005,2026-09-01,Fresher,Unknown,Mentor Session,1,999,0.0,Failed,2026,9,September,Q3,999,0.00,999.00
6,RT-1006,2026-10-13,Student,Pune,Learning Kit,2,799,10.0,Paid,2026,10,October,Q4,1598,159.80,1438.20
7,RT-1007,2026-12-01,Professional,Mumbai,Mentor Session,2,999,0.0,Paid,2026,12,December,Q4,1998,0.00,1998.00



BEFORE VS AFTER
Original rows: 12
Final rows: 6
Original columns: 9
Final columns: 16
Original missing values: 3
Final missing values: 0

CLEANING PROCESS FINISHED
CSV file created: True
Output file: /content/clean_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Dataset Limitation:** The dataset provided by RabTech Academy contains 12 records and 9 original columns. All cleaning and preprocessing steps were performed on the available dataset. The original dataset was not artificially expanded. Therefore, the 10,000-row requirement could not be met using the supplied data alone.

In [32]:
import pandas as pd

df = pd.read_csv("/content/retail-orders-raw.csv")

print(df["order_date"].to_string(index=False))

2026-01-03
03/01/2026
2026-01-05
2026-01-07
2026-01-07
2026-01-09
2026-13-10
2026-01-12
2026-01-14
2026-01-16
2026-01-18
       NaN


In [33]:
from google.colab import files
files.download("/content/clean_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>